In [0]:
dbutils.widgets.text("catalog","dev")

In [0]:
catalog = dbutils.widgets.get("catalog")

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.bronze.Last_Run(
    Table_Name STRING,
    Last_Run TIMESTAMP
)
""")

In [0]:
# %sql
# INSERT INTO dev.bronze.last_run(Table_Name, Last_Run) 
#     VALUES('customers', '1900-01-01'),
#           ('order_items', '1900-01-01'),
#           ('orders', '1900-01-01'),
#           ('products', '1900-01-01')

In [0]:
# Define all tables with their primary keys
table_configs = [
    {
        "table_name":   "customers",
        "primary_key":  "customer_id"
    },
    {
        "table_name":   "products",
        "primary_key":  "product_id"
    },
    {
        "table_name":   "orders",
        "primary_key":  "order_id"
    },
    {
        "table_name":   "order_items",
        "primary_key":  "order_item_id"
    }
]

In [0]:
for config in table_configs:
    table_name   = config["table_name"]
    primary_key  = config["primary_key"]
  
    last_run_time = spark.sql(f"""
        SELECT last_run FROM {catalog}.bronze.last_run
        WHERE table_name = '{table_name}'
        """).collect()[0][0]
  
    #Getting the Incremental Data
    df_incremental = spark.read.table(f"{catalog}.bronze.{table_name}") \
        .filter(
            (col("created_at") > last_run_time) | 
            (col("updated_at") > last_run_time)
        )

    #DeDuplication
    window = Window.partitionBy(primary_key).orderBy(desc("updated_at"))
    df_incremental= df_incremental.withColumn("rn", row_number().over(window)) \
                .filter(col("rn") == 1) \
                .drop("rn")
  
    columns = [c for c in df_incremental.columns if c != primary_key]
    set_clause    = ", ".join([f"target.{c} = source.{c}" for c in columns])
    insert_cols   = ", ".join([primary_key] + columns)
    insert_vals   = ", ".join([f"source.{c}" for c in [primary_key] + columns])
    
    #One time for table creations 
    #df_incremental.write.mode("overwrite").saveAsTable(f"{catalog}.silver.{table_name}")

    #Creating the View to be used in MERGE
    df_incremental.createOrReplaceTempView("source_data")

    #Incremental Logic
    spark.sql(f"""
        MERGE INTO {catalog}.silver.{table_name} AS target
        USING source_data AS source
        ON target.{primary_key} = source.{primary_key}

        WHEN MATCHED AND source.updated_at > target.updated_at THEN
            UPDATE SET {set_clause}

        WHEN NOT MATCHED THEN
            INSERT ({insert_cols})
            VALUES ({insert_vals})
        """)

    #Updating the Last Run Table    
    new_lastrun = df_incremental.select(
        greatest(max("created_at"), max("updated_at"))
    ).collect()[0][0]

    if new_lastrun is not None:
        spark.sql(f"""
            MERGE INTO {catalog}.bronze.last_run AS target
            USING (SELECT '{table_name}' AS table_name,
                   CAST('{new_lastrun}' AS TIMESTAMP) AS last_run) AS source
            ON target.table_name = source.table_name
            WHEN MATCHED THEN
                UPDATE SET target.last_run = source.last_run
        """)

In [0]:
df_customers = spark.read.table(f"{catalog}.silver.customers")
df_orders = spark.read.table(f"{catalog}.silver.orders")
df_products = spark.read.table(f"{catalog}.silver.products")
df_order_items = spark.read.table(f"{catalog}.silver.order_items")
df_customers_orders = df_orders.join(broadcast(df_customers), on="customer_id", how="inner")
df_product_orders = df_order_items.join(broadcast(df_products), on="product_id", how="inner")

In [0]:
display(df_product_orders)

In [0]:
# Select unique columns by qualifying ambiguous references
df_customers_orders.select(
    "customer_id","order_id","order_date","total_amount","order_status",
    col("city").alias("Customer_City"),
    col("state").alias("Customer_State")
).write.mode("overwrite").saveAsTable(f"{catalog}.silver.customer_orders")

df_product_orders.select(
    "product_id","order_item_id","order_id","quantity",
    col("order_items.price").alias("price"),
    col("product_name").alias("Product_Name"),
    col("category").alias("Product_Category"),
    col("products.price").alias("Product_Price")
).write.mode("overwrite").saveAsTable(f"{catalog}.silver.product_orders")